# Chapter 1: Fundamentals of AI Agents

Estimated time: about 6 hours. Roughly 30-45 minutes of that is one-time account/API setup,
longer than usual on purpose, since it's written for a reader with limited Python
background; the remaining 5+ hours is concept, build, break-it, and the interview drill.

Prerequisites: none beyond the repo-level Prerequisites gut-check in the root
`README.md`. This chapter assumes zero prior agent experience and doesn't assume you're
comfortable with a terminal, `pip`, or environment variables yet.

Interview category this chapter maps to: the baseline vocabulary and judgment questions
almost every AI-agent-engineering interview opens with: "what is an agent, really," ReAct
basics, and "your agent is stuck in a loop, what do you do."


## Concept: what an agent actually is

Think of an AI agent as a new employee on their first day. It's given a task, it has
access to some company resources (tools: a calculator, a search system, whatever's
relevant), and it's expected to use them, check the results, and report back. A bad new
employee gets stuck on a task and just keeps trying the same thing forever instead of
escalating. A good one knows when to stop, ask for help, or say "I'm done." This chapter is
effectively that employee's training day; everything here is a warm-up for the multi-agent
"small team" you'll build in Chapter 2.

That metaphor is intuition-building, not a replacement for the real vocabulary. Here's the
vocabulary an interviewer actually expects you to use.

### Chatbot vs. workflow vs. agent

These three get conflated constantly, and "explain the difference" is a very common opening
interview question.

| | Chatbot | Workflow | Agent |
|---|---|---|---|
| Control flow | Fixed: user asks, model replies, repeat | Fixed, predetermined sequence of steps written by a developer | Dynamic: the model decides what to do next at each step |
| Tool use | None, or a single hardcoded call | Deterministic calls to specific tools in a fixed order | The model chooses which tool(s) to call and when, based on what it observes |
| Autonomy | None: always waits for the next human turn | None: the sequence never deviates, regardless of what happens | Can take several actions in a row with no human turn in between |
| Employee framing | Someone at a help desk, one question at a time | Someone following a fixed checklist, step 1 through step N, no deviation | Someone given a goal who decides which resources to use, and in what order |
| Typical failure mode | Doesn't remember earlier turns; can't take action | Breaks the moment reality doesn't match the checklist | Can loop forever or wander off-task without proper guardrails (see this chapter's "break it" section) |

### The ReAct pattern

Most agent loops, including the one you'll build below, follow ReAct: reason, then act (Yao
et al., 2022; see `REFERENCES.md`). Concretely, this means interleaving three things in a
loop. First, a thought: the model reasons about what to do next, in its own words. Then an
action: it picks a tool and produces structured input for it. This is "tool calling" or
"function calling"; the model isn't executing anything itself, it's requesting that your
code execute something and hand the result back. Finally, an observation: your code runs the
tool and feeds the result back into the model's context, and the loop repeats until the
model decides it has enough to answer.

### When *not* to use an agent

Would you actually hire someone for this role, or would a simple checklist, or a vending
machine, do the job? If a task is fully deterministic (same input always produces the same
correct output via fixed logic), latency- or cost-sensitive in a way an LLM call can't meet,
or a high-stakes irreversible action that genuinely needs a human decision, an agent is very
often the wrong tool. Chapter 8 covers this judgment call in much more depth; this is the
seed of it.

### Core vocabulary

A token is roughly a word-piece, not quite a word, not quite a character. The context window
is the maximum number of tokens a model can attend to at once, covering the system prompt,
conversation history, and everything generated so far.

Tool or function calling means the model produces a structured request, such as calling
`calculator` with `{"expression": "12 * 7"}`, instead of free text; your code is what
actually executes it.

Grounding means anchoring a model's output in specific, checkable source material (a
document, a tool result) rather than relying purely on what it learned during training.

Hallucination is the model confidently producing content that's false, unsupported, or
invented. Grounding reduces hallucination risk; it does not eliminate it (Chapter 3 shows
exactly why, hands-on).


## Setup

A small bit of bootstrapping so this notebook works the same way whether you open it from
the repo root or from inside `curriculum/` (Jupyter's working directory varies depending on
how you launch it), plus a fixed random seed: every notebook in this course seeds its
randomness so outputs are reproducible across machines and CI runs.


In [1]:
import sys
from pathlib import Path

# Make `agentlib` importable regardless of this notebook's working directory.
_repo_root = Path.cwd()
if not (_repo_root / "agentlib").is_dir():
    _repo_root = _repo_root.parent
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

import json
import random
import re

from agentlib.grading import check

random.seed(42)
print(f"Repo root on sys.path: {_repo_root}")


Repo root on sys.path: /home/user/learning-agentic-ai


## Build: giving the new employee some company resources

Every employee needs tools to do their job. We'll give ours two: a calculator and a search
system. Both are deterministic and fully offline, so this chapter's output never depends on
network state or model randomness; that comes later, once a real model is plugged in.

The calculator deliberately does not use Python's `eval()` on the raw string a model
produces, because that would let arbitrary code execution slip in through model output,
which is a real production security concern (input from an LLM should be treated the same as
any other untrusted input). Instead it parses the expression into an AST and only allows
numbers and basic arithmetic operators.


In [2]:
import ast
import operator

_ALLOWED_OPERATORS = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.Pow: operator.pow,
    ast.USub: operator.neg,
    ast.UAdd: operator.pos,
}


def _eval_node(node):
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _ALLOWED_OPERATORS:
        return _ALLOWED_OPERATORS[type(node.op)](_eval_node(node.left), _eval_node(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _ALLOWED_OPERATORS:
        return _ALLOWED_OPERATORS[type(node.op)](_eval_node(node.operand))
    raise ValueError(f"Unsupported expression element: {ast.dump(node)}")


def calculator_tool(expression: str) -> dict:
    '''The 'calculator' resource: a restricted-eval arithmetic tool. Only numbers and
    +, -, *, /, ** are allowed -- anything else (names, calls, imports, ...) is rejected
    rather than executed, since this tool's input may come from model output.'''
    try:
        tree = ast.parse(expression, mode="eval")
        result = _eval_node(tree.body)
        return {"status": "ok", "result": result}
    except Exception as exc:
        return {"status": "error", "error": str(exc)}


# quick sanity check
print(calculator_tool("12 * 7"))
print(calculator_tool("(3 + 4) * 2 - 1"))
print(calculator_tool("__import__('os').listdir()"))  # rejected, not executed


{'status': 'ok', 'result': 84}
{'status': 'ok', 'result': 13}
{'status': 'error', 'error': "Unsupported expression element: Call(func=Attribute(value=Call(func=Name(id='__import__', ctx=Load()), args=[Constant(value='os')], keywords=[]), attr='listdir', ctx=Load()), args=[], keywords=[])"}


Now the search resource, a small, deterministic, canned-fact lookup. A real search tool
hits a live index; this one hits a fixed dictionary so the rest of this chapter's output
never depends on the internet being up.


In [3]:
_MOCK_FACTS = {
    "anthropic founder": (
        "Anthropic was founded in 2021 by Dario Amodei and Daniela Amodei, along with "
        "several colleagues who had previously worked at OpenAI."
    ),
    "react pattern": (
        "ReAct interleaves reasoning traces with actions, letting a model plan and use "
        "tools within the same loop (Yao et al., 2022)."
    ),
    "context window": (
        "A context window is the maximum number of tokens a model can attend to at once, "
        "including the prompt, conversation history, and its own output so far."
    ),
}


def mock_search_tool(query: str) -> dict:
    '''The 'search' resource: a small, deterministic, canned-fact lookup.'''
    q = query.lower()
    for key, fact in _MOCK_FACTS.items():
        if key in q or any(word in q for word in key.split()):
            return {"status": "ok", "result": fact}
    return {"status": "not_found", "result": f"No canned result for query: {query!r}"}


print(mock_search_tool("who founded anthropic?"))
print(mock_search_tool("weather in tokyo"))


{'status': 'ok', 'result': 'Anthropic was founded in 2021 by Dario Amodei and Daniela Amodei, along with several colleagues who had previously worked at OpenAI.'}
{'status': 'not_found', 'result': "No canned result for query: 'weather in tokyo'"}


## The "brain": a stand-in for a real model

Before wiring up a real model call (that comes later, in the setup section below), we need
something to play the role of "the model deciding what to do." `fake_llm_brain()` below is
explicitly a stand-in, not a real LLM: it's a small rule-based function that looks at what's
happened so far and picks the next action via simple keyword/state checks. It exists so the
ReAct loop's *mechanics* can be taught and demonstrated for free, fully offline, with fully
deterministic output. It is approximating what a real model does; it is not doing what a
real model does.


In [4]:
def fake_llm_brain(messages: list) -> dict:
    '''Rule-based stand-in for a real LLM 'brain' -- NOT reasoning, just deterministic
    keyword/state matching. Returns one of:
      {"action": "calculator",  "action_input": "<expression>"}
      {"action": "mock_search", "action_input": "<query>"}
      {"action": "final_answer","action_input": "<text>"}
    '''
    task_text = messages[0]["content"].lower()
    observations = [m for m in messages if m["role"] == "observation"]

    has_math = bool(re.search(r"\d+\s*[+\-*/]\s*\d+", task_text))
    wants_search = "anthropic" in task_text and any(
        w in task_text for w in ["who", "found", "search"]
    )

    did_math = any(o["tool"] == "calculator" for o in observations)
    did_search = any(o["tool"] == "mock_search" for o in observations)

    if has_math and not did_math:
        expr = re.search(r"[\d.\s+\-*/()]{3,}", task_text).group().strip()
        return {"action": "calculator", "action_input": expr}
    if wants_search and not did_search:
        return {"action": "mock_search", "action_input": "anthropic founder"}

    parts = [str(o["content"].get("result", o["content"])) for o in observations]
    answer = " ".join(parts) if parts else "I don't have enough information to answer."
    return {"action": "final_answer", "action_input": answer}


## Build: the minimal ReAct loop

This is the actual Thought -> Action -> Observation loop, and it is yours to write: ask the
brain what to do, run the tool it picked, feed the result back in, repeat until it says it's
done (or you hit an iteration cap).

The one part worth thinking about before you type is the *feed the result back in* step. The
brain is stateless; the only thing it knows about what has already happened is what it can
read in `messages`. An observation that never gets appended is an observation the brain never
sees, so it will keep asking for the same thing forever.

Every graded cell in this course ends with `your_function = check("task-id", your_function)`.
That call runs a suite of assertions against what you wrote and, if any fail, prints exactly
which ones and why; it hands your function straight back, so the rebinding is a no-op and
later cells just use the name as normal. Run `python grade.py` at any point to see where you
stand.

In [ ]:
TOOLS = {
    "calculator": calculator_tool,
    "mock_search": mock_search_tool,
}


def run_agent(task: str, brain, tools: dict, max_iterations: int = 6, verbose: bool = True) -> str:
    '''Run the ReAct loop until the brain gives a final answer or the budget runs out.

    brain(messages) -> {"action": ..., "action_input": ...}, where action is either
    "final_answer" or a key of `tools`. Seed `messages` with the task as a user turn:

        {"role": "user", "content": task}

    and append every tool result back onto it as:

        {"role": "observation", "tool": <action>, "content": <what the tool returned>}

    Return the final answer's action_input. If the brain names a tool that doesn't exist,
    return a message saying so rather than raising; if the budget runs out, return a message
    saying that instead.
    '''
    raise NotImplementedError("Implement me, then re-run this cell")


run_agent = check("ch01-react-loop", run_agent)

In [ ]:
result = run_agent("What is 12 * 7? Also, who founded Anthropic?", fake_llm_brain, TOOLS)
print("\n=== RESULT ===")
print(result)

## Build: memory on vs. memory off

Separate from the ReAct tool loop above, a real agent also has to manage *conversation*
memory across turns: whether earlier turns get threaded back into the prompt at all. There is
no separate memory mechanism unless you build one; "memory" is conversation history being
concatenated back into context on every call, which means it is your job to put it there.

You write the threading. The brain below is given, and it can only recall what it can read in
the history it is handed.

In [ ]:
def fake_llm_brain_chat(history: list, user_input: str) -> str:
    '''Toy conversational stand-in for this memory demo only -- separate from
    fake_llm_brain's ReAct loop above, since conversational memory and within-task tool
    memory are two different things a real agent has to manage. It recalls a name only if
    the name is somewhere in the history it is given.'''
    context_text = " ".join(m["content"] for m in history)
    if "what is my name" in user_input.lower():
        match = re.search(r"my name is (\w+)", context_text, re.IGNORECASE)
        if match:
            return f"Your name is {match.group(1)}."
        return "I don't know your name -- you haven't told me, or I'm not remembering earlier turns."
    return "Noted."


def chat_turn(history: list, user_input: str, brain) -> str:
    '''Run one conversational turn and thread it into `history`.

    Call brain(history, user_input) -- the brain sees the conversation as it stood BEFORE
    this turn, plus the new input. Then record both halves of the exchange onto `history`
    in place, as:

        {"role": "user", "content": user_input}
        {"role": "assistant", "content": <the reply>}

    Return the reply.

    Both halves. Recording only the agent's own replies is the classic half-implementation:
    everything the user actually told you disappears.
    '''
    raise NotImplementedError("Implement me, then re-run this cell")


chat_turn = check("ch01-memory", chat_turn)

In [ ]:
history = []

turn1 = "My name is Jack."
print("User:", turn1)
print("Agent:", chat_turn(history, turn1, fake_llm_brain_chat))

turn2 = "What is my name?"
print("\nUser:", turn2)
# Memory OFF is not a separate mode -- it is simply not passing the history along.
print("Agent (memory OFF):", fake_llm_brain_chat([], turn2))
print("Agent (memory ON): ", fake_llm_brain_chat(history, turn2))

With memory off, the earlier turn never gets threaded back into what the model sees. It's
answering from nothing, the same way a new employee would if every request landed on their
desk with zero context about the conversation so far. In a real prompt, "memory" is just
conversation history being concatenated back into context on every call; there's no separate
memory mechanism unless you build one (Chapter 4 covers caching and staleness in that
history).


## Break it: the employee who never escalates

Here's the failure mode this chapter is really about: an employee assigned a task who gets
stuck and just... keeps trying the same thing, forever, without ever stopping to say "this
isn't working, I need to ask for help." We'll build a "bait tool" that always reports it's
not done yet, and a brain with no escalation logic, and watch what happens.

In [7]:
def bait_tool(_input):
    '''Always reports 'still working on it', regardless of input -- models a system (or
    an employee) that never actually finishes and never escalates.'''
    return {"status": "partial", "detail": "Still processing your request, check back."}


def looping_brain(messages):
    '''No escalation logic: if the last observation says the task isn't done, ask again.
    This is the bug.'''
    observations = [m for m in messages if m["role"] == "observation"]
    if not observations or observations[-1]["content"].get("status") == "partial":
        return {"action": "bait_tool", "action_input": "any"}
    return {"action": "final_answer", "action_input": "done"}


print("--- Demonstrating the bug: no stop condition, no escalation ---\n")
buggy_result = run_agent(
    "Generate my quarterly report.",
    looping_brain,
    tools={"bait_tool": bait_tool},
    max_iterations=20,  # capped ONLY so this notebook doesn't hang -- see note below
    verbose=True,
)
print("\n=== RESULT ===")
print(buggy_result)


--- Demonstrating the bug: no stop condition, no escalation ---

[step 1] Thought -> Action: bait_tool('any')
[step 1] Observation: {'status': 'partial', 'detail': 'Still processing your request, check back.'}
[step 2] Thought -> Action: bait_tool('any')
[step 2] Observation: {'status': 'partial', 'detail': 'Still processing your request, check back.'}
[step 3] Thought -> Action: bait_tool('any')
[step 3] Observation: {'status': 'partial', 'detail': 'Still processing your request, check back.'}
[step 4] Thought -> Action: bait_tool('any')
[step 4] Observation: {'status': 'partial', 'detail': 'Still processing your request, check back.'}
[step 5] Thought -> Action: bait_tool('any')
[step 5] Observation: {'status': 'partial', 'detail': 'Still processing your request, check back.'}
[step 6] Thought -> Action: bait_tool('any')
[step 6] Observation: {'status': 'partial', 'detail': 'Still processing your request, check back.'}
[step 7] Thought -> Action: bait_tool('any')
[step 7] Observation

Twenty identical "still processing" observations in a row, and the loop only stopped
because we artificially capped `max_iterations` at 20 purely so this notebook wouldn't hang
forever. A real production agent has no such cap by default, which is exactly how a runaway
agent burns unbounded tokens (and money) on a bug like this. Chapter 1's setup section below
has you set a real spend limit specifically as a guardrail against this class of bug.

The fix is duplicate-observation detection, and it's yours to write: if the exact same
observation fires twice in a row with no progress, stop and escalate instead of continuing.
That's the equivalent of an employee saying "I've asked twice and gotten the same
non-answer both times, I need to flag this" instead of asking a third, fourth, fifth time.

Note the exact rule: *twice in a row*. Not "an observation I've seen at some point" -- an
agent that revisits an earlier state after doing real work in between is making progress,
and a guard that remembers every observation forever would kill it for no reason.


In [ ]:
def run_agent_with_guard(task: str, brain, tools: dict, max_iterations: int = 50, verbose: bool = True) -> str:
    '''Same loop as run_agent(), but escalates instead of repeating once the exact same
    observation fires twice IN A ROW.

    Everything run_agent() does still applies: seed messages with the task, run the tool the
    brain names, thread each observation back. The addition is a comparison against the
    previous observation only -- if this one is identical, stop and return a message
    explaining that you are escalating (the word "escalating" should appear in it) instead
    of appending the observation and looping again.

    Tool results are dicts, so compare them by value, not by identity.
    '''
    raise NotImplementedError("Implement me, then re-run this cell")


run_agent_with_guard = check("ch01-dup-guard", run_agent_with_guard)

In [ ]:
print("--- Demonstrating the fix: duplicate-observation detection ---\n")
fixed_result = run_agent_with_guard(
    "Generate my quarterly report.",
    looping_brain,
    tools={"bait_tool": bait_tool},
    max_iterations=50,
    verbose=True,
)
print("\n=== RESULT ===")
print(fixed_result)

Before: 20 iterations of identical spam, stopped only by an artificial cap. After: 2 steps,
then a clear escalation message. Same buggy brain, same bait tool; the only difference is
the guard.


## Account, budget, and API key setup

Everything so far ran entirely offline with fake tools and a fake brain. This section is
where you connect this course to a real model provider. It's the one-time setup that
Chapters 2, 3, 4, 6, 7, and the capstone all reuse, so it's worth doing carefully now.

This section assumes you haven't necessarily used a terminal, `pip`, or environment
variables before. If you already have, skim it and move on; if you haven't, each step below
says what it is and why it matters, not just the command to run.

### Step 1: Choose a provider

You have two options: Anthropic (`console.anthropic.com`) or OpenAI
(`platform.openai.com`). Either works throughout this entire course. The choice gets stored
in one environment variable (`LLM_PROVIDER`), and nothing else in the curriculum needs to
know which one you picked. If you don't already have a preference, either is a fine default;
pick whichever you're more likely to use professionally.

### Step 2: Create an account

Sign up at whichever console you chose. New accounts sometimes receive a small trial credit;
don't count on a specific amount, since offers change, but it's common to get a small amount
of free credit to start with.

### Step 3: Set a spend limit before you generate a key

Do this before step 4, not after. A spend limit is a hard ceiling on how much you can be
charged, independent of anything your code does. It's the single most important step in this
whole section, given that Chapters 1 and 2 are literally about teaching you to detect a
runaway agent loop. A limit means a bug can't turn into a surprise bill while you're still
learning to catch bugs like that.

On Anthropic: in the console, go to Settings → Billing, add a payment method, and set a
spending limit there. If that exact menu label has moved by the time you read this (console
UIs change), look under Billing/Usage for a spend or budget control, or simply leave
auto-reload disabled on prepaid credits, which caps your maximum possible spend at whatever
credit balance you've purchased, with no extra menu-hunting required.

On OpenAI: in the platform dashboard, go to your project's Limits page and set a monthly
budget there. Same caveat: if the exact path has moved, look for "Limits" or "Usage limits"
under your project or organization settings.

Recommended ceiling: $15 to $20 for the entire course. That's a comfortable margin. A full
pass through the course, including reasonable re-running of cells while learning,
realistically costs $5 to $15 on either provider's cheapest current-generation model (see the
cost note below). Console UIs change often, so confirm the exact current menu path against
your live dashboard rather than trusting this paragraph blindly.

### Step 4: Generate an API key

Once your spend limit is set, generate an API key from the same console.

### Step 5: Create a `.env` file

In the root of this repo, copy `.env.example` to a new file named `.env` and fill in:

```
LLM_PROVIDER=anthropic
ANTHROPIC_API_KEY=sk-ant-...
```

(or `LLM_PROVIDER=openai` with `OPENAI_API_KEY=...`, if that's what you picked). A `.env`
file is just a plain-text list of `NAME=value` pairs that gets loaded into your program's
environment at runtime. It's the standard way to keep secrets like API keys out of your
actual code and out of git.

### Step 6: Never commit `.env`

This repo's `.gitignore` already excludes `.env`, so `git add .` will never accidentally
stage it. Still worth knowing why: an API key committed to git history is effectively public
forever, even if you delete it in a later commit, because git remembers.

### Step 7: Install the SDKs

If you're working in this repo, `pip install -r requirements.txt` (from the root README's
setup instructions) already installs both `anthropic` and `openai`, regardless of which
provider you picked, so switching providers later never requires a fresh install.

### A note on cost

Real-API calls are the default across most of this course. Chapters 1, 2, 3, 4, 6, 7, and
the capstone all call a real model by default once this setup is done, falling back to a
mock/offline path (like everything above this section) only when no key is present. As of
this repo's last verification date (see `REFERENCES.md`):

Anthropic's Claude Haiku 4.5, this course's default model, is $1/$5 per million
input/output tokens. Anthropic's Claude Sonnet 5, the stronger tier some chapters use, is
$2/$10 per million tokens; Anthropic made this permanent on 2026-08-11, rather than the
price increase to $3/$15 that had originally been scheduled for September 1, 2026. OpenAI's
GPT-5.6 Luna, this course's default OpenAI model, has had inconsistently reported pricing
across sources during this repo's construction (note that the bare `"gpt-5.6"` alias
currently routes to a different, more expensive tier, so use the exact model ID
`gpt-5.6-luna`). Rather than assert a number here that's likely to be stale, check
`https://developers.openai.com/api/docs/pricing` for the current figure.

Per-token pricing on both platforms changes often; check the provider's live pricing page
before trusting any number written into a notebook months or years after it was built.

### No key, no budget? Use Ollama instead

If you'd rather not create an account or spend anything at all, you can get real (if
smaller/weaker) model behavior for free by running a model locally via
[Ollama](https://ollama.com): install it, run `ollama pull llama3.2` (or any model you like)
from a terminal, and it serves an OpenAI-compatible API on `localhost:11434` with no account,
no key, and no spend limit needed at all. This course's `agentlib.llm_client` doesn't wire
this up automatically. It's a pointer for the curious, not a third code path this repo
maintains, but if you want to try it, point an OpenAI-compatible client at that local
endpoint instead of `platform.openai.com`.


In [9]:
from agentlib.llm_client import DEFAULT_MODELS, HAS_KEY, LLM_PROVIDER, STRONG_MODELS

print(f"LLM_PROVIDER = {LLM_PROVIDER!r}")
print(f"HAS_KEY      = {HAS_KEY}")

if HAS_KEY:
    print(f"Real API calls will use: {DEFAULT_MODELS[LLM_PROVIDER]} (default) "
          f"/ {STRONG_MODELS[LLM_PROVIDER]} (stronger tier, used by later chapters)")
else:
    print("No API key found -- this notebook will fall back to the deterministic mock brain "
          "built above. Complete the setup section above (or use Ollama) to try the real path.")


LLM_PROVIDER = 'anthropic'
HAS_KEY      = False
No API key found -- this notebook will fall back to the deterministic mock brain built above. Complete the setup section above (or use Ollama) to try the real path.


## Build: a real brain, on top of a provider-agnostic client

`HAS_KEY` and `LLM_PROVIDER` above come from `agentlib/llm_client.py`, the one shared
library module this course builds directly in `agentlib` from Chapter 1 onward, rather than
inline-first like the tools and loop guards above. Every other real-API section from Chapter
2 onward imports it unchanged.

That's deliberate, and worth calling out as a real skill in its own right: building directly
against one vendor's SDK everywhere is a common early mistake that creates painful lock-in
later. `agentlib.llm_client.call_model()` reads `LLM_PROVIDER` once and normalizes both
providers' responses, including their different tool-calling formats, into one common shape,
so the rest of this course (and, ideally, your own future projects) never has to branch on
which provider is active.

Below, `RealLLMBrain` is backed by a real model call through that shared client, with proper
tool-use schemas for the calculator and search resources, matching `fake_llm_brain`'s exact
`brain(messages) -> {"action": ..., "action_input": ...}` interface, so `run_agent()` above
can use either brain interchangeably.


In [10]:
from agentlib import llm_client

CALCULATOR_TOOL_SCHEMA = {
    "name": "calculator",
    "description": "Evaluate a basic arithmetic expression (+, -, *, /, **). Input must be "
                    "a plain arithmetic expression string, e.g. '12 * 7'.",
    "input_schema": {
        "type": "object",
        "properties": {"expression": {"type": "string"}},
        "required": ["expression"],
    },
}

MOCK_SEARCH_TOOL_SCHEMA = {
    "name": "mock_search",
    "description": "Search a small internal knowledge base for a fact. Input is a short "
                    "query string.",
    "input_schema": {
        "type": "object",
        "properties": {"query": {"type": "string"}},
        "required": ["query"],
    },
}

AGENT_SYSTEM_PROMPT = (
    "You are a helpful employee with access to a calculator and a search tool. Use them "
    "when needed to answer the user's request accurately, then give a final answer in "
    "plain text with no further tool calls."
)


class RealLLMBrain:
    '''Stateful wrapper around agentlib.llm_client.call_model() that maintains proper
    provider-native tool-call turn structure across a single agent run, while matching
    fake_llm_brain's simple brain(messages) -> action-dict interface for run_agent().'''

    def __init__(self, model: str | None = None):
        self.model = model or llm_client.DEFAULT_MODELS[llm_client.LLM_PROVIDER]
        self.provider_messages: list[dict] = []
        self._last_response = None
        self._last_tool_call = None

    def __call__(self, messages: list) -> dict:
        if not self.provider_messages:
            self.provider_messages.append({"role": "user", "content": messages[0]["content"]})
        else:
            latest_observation = messages[-1]
            self.provider_messages.append(llm_client.format_assistant_tool_call(self._last_response))
            self.provider_messages.append(
                llm_client.format_tool_result(
                    self._last_tool_call, json.dumps(latest_observation["content"])
                )
            )

        response = llm_client.call_model(
            messages=self.provider_messages,
            system=AGENT_SYSTEM_PROMPT,
            tools=[CALCULATOR_TOOL_SCHEMA, MOCK_SEARCH_TOOL_SCHEMA],
            model=self.model,
        )
        self._last_response = response

        if response.tool_calls:
            tc = response.tool_calls[0]
            self._last_tool_call = tc
            key = "expression" if tc.name == "calculator" else "query"
            return {"action": tc.name, "action_input": tc.input.get(key, "")}
        return {"action": "final_answer", "action_input": response.text}


### The toggle

This is the line that makes the real API the default experience rather than an optional
add-on: when a key is present, every run below uses a real model; when it isn't (as in CI,
or for a learner without budget), it transparently falls back to the deterministic mock
brain from earlier in this chapter: same `run_agent()` function, same interface, zero code
changes required either way.


In [11]:
brain = RealLLMBrain() if HAS_KEY else fake_llm_brain
print(f"Using: {'RealLLMBrain (real API call)' if HAS_KEY else 'fake_llm_brain (mock, no key present)'}")

final = run_agent("What is 12 * 7? Also, who founded Anthropic?", brain, TOOLS)
print("\n=== RESULT ===")
print(final)


Using: fake_llm_brain (mock, no key present)
[step 1] Thought -> Action: calculator('12 * 7')
[step 1] Observation: {'status': 'ok', 'result': 84}
[step 2] Thought -> Action: mock_search('anthropic founder')
[step 2] Observation: {'status': 'ok', 'result': 'Anthropic was founded in 2021 by Dario Amodei and Daniela Amodei, along with several colleagues who had previously worked at OpenAI.'}
[step 3] Thought -> Action: final_answer('84 Anthropic was founded in 2021 by Dario Amodei and Daniela Amodei, along with several colleagues who had previously worked at OpenAI.')
[step 3] Final answer: 84 Anthropic was founded in 2021 by Dario Amodei and Daniela Amodei, along with several colleagues who had previously worked at OpenAI.

=== RESULT ===
84 Anthropic was founded in 2021 by Dario Amodei and Daniela Amodei, along with several colleagues who had previously worked at OpenAI.


## Interview preparation

### Recap

- An agent differs from a chatbot or a workflow by *dynamic* control flow: the model decides
  what to do next, not just what to say next.
- ReAct (Thought -> Action -> Observation) is the loop shape underneath most agents,
  including the one you just built.
- An agent with no stop condition and no escalation logic can loop forever, burning
  unbounded cost. Duplicate-observation detection is one concrete, cheap guard against it
  (Chapter 2 builds a proper max-iteration guard and a harder-to-catch cycle variant).
- Grounding reduces hallucination risk, but it does not eliminate it.
- Building against a provider-agnostic client rather than one vendor's SDK directly is a
  real production practice, not just a teaching convenience.

### Vocabulary flashcards (optional, interactive)

Skippable: if you're not running this in an interactive terminal/Jupyter session (e.g. this
cell is being executed headlessly in CI), it will detect that and skip itself automatically.


In [12]:
FLASHCARDS = [
    ("Context window", "The maximum number of tokens (prompt + history + output-so-far) a model can attend to at once."),
    ("Tool / function calling", "A model producing a structured request to invoke an external function, rather than free text -- your code does the actual invoking."),
    ("Grounding", "Anchoring a model's output in specific, checkable source material rather than relying purely on parametric knowledge."),
    ("Hallucination", "A model confidently producing content that's false, unsupported, or invented."),
    ("ReAct", "Thought -> Action -> Observation: an agent pattern interleaving reasoning traces with tool actions (Yao et al., 2022)."),
    ("Token", "A chunk of text, often a sub-word piece, that's the model's basic unit of input and output."),
]


def flashcard_quiz(cards: list, interactive: bool = True) -> None:
    if not interactive:
        print("Quiz skipped (interactive=False).")
        return
    for term, definition in cards:
        try:
            input(f"Define: {term}\n> ")
        except Exception:
            # Covers both a plain EOFError (stdin closed) and Jupyter's
            # StdinNotImplementedError (headless kernel, e.g. this notebook running in CI) --
            # either way, there's no one there to type an answer, so skip gracefully.
            print(
                "\n(No interactive input available -- skipping the rest of the quiz. Run "
                "this cell in a real terminal or Jupyter session to try it for real.)"
            )
            return
        print(f"Reference definition: {definition}\n")
    print("Quiz complete.")


flashcard_quiz(FLASHCARDS, interactive=True)



(No interactive input available -- skipping the rest of the quiz. Run this cell in a real terminal or Jupyter session to try it for real.)


### Explain it to a non-technical PM

Write, in 3-5 plain-English sentences, how you'd explain what an AI agent is to a product
manager who's never used one, without jargon (no "ReAct," "tool calling," "context window").
The employee metaphor from this chapter's concept section is fair game if it helps.


In [13]:
my_pm_explanation = '''
(Write your answer here.)
'''

print(my_pm_explanation)



(Write your answer here.)



### Cold-answer questions

Attempt these from memory before checking `solutions/ch01_fundamentals_answers.md`. These
map directly to real interview phrasings.

1. Your AI agent keeps looping forever. How would you detect and stop it?
2. A stakeholder says "we don't need an agent here, a simple script would do." When are they
   right?
3. Explain the difference between a chatbot, a workflow, and an agent to a non-technical
   colleague.
4. What's the difference between a model's context window and "memory" across a
   conversation?
5. Why can a RAG-grounded model still hallucinate?

Check your answers against `solutions/ch01_fundamentals_answers.md`.

## Next: Chapter 2: Agent Control Flow

This chapter built one employee working alone. Chapter 2 turns that into a small team, Jack
(planner), Bob (worker), and Mike (critic), and introduces subagents: giving a teammate a
bounded task with a fresh, isolated context, and getting back a compressed summary instead
of their entire raw train of thought. It also covers a harder-to-catch failure mode than the
straight-line loop you just fixed: a genuine *cycle*, where two employees defer the same
decision back and forth without ever resolving it.
